#🧩 Step 1: Install Required Libraries for RAG Implementation

Before building a Retrieval-Augmented Generation (RAG) system, we need to install essential Python packages that handle language model operations, embeddings, and document processing.


In [3]:
# 🧩 Step 1: Install Required Libraries for RAG Implementation

# The following command installs and upgrades the necessary libraries:
# - langchain: main LLM orchestration framework
# - langchain-community: community extensions and integrations
# - langchain-openai: OpenAI API integration for embeddings and chat models
# - langchain-pinecone: vectorstore connector for Pinecone
# - pinecone: official Python client to interact with Pinecone database
# - pypdf: PDF parser to extract text from documents
# - tiktoken: tokenizer to manage and count tokens for OpenAI models

!pip -q install -U \
  langchain langchain-community==0.2.* langchain-openai \
  langchain-pinecone==0.1.* pinecone==4.* pypdf==4.* tiktoken langsmith[openai-agents]


In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#🔐 Step 2: Set Up Environment Variables and API Keys

In this step, we configure the necessary environment variables for our RAG workflow.
These environment variables store authentication credentials and configuration settings for external services that LangChain and Pinecone rely on.


In [8]:
# 🔐 Step 2: Set Up Environment Variables and API Keys

# Import the Colab `userdata` module to securely fetch stored secrets.
from google.colab import userdata
import os

# Enable LangSmith tracing for detailed observability of LangChain runs.
os.environ["LANGSMITH_TRACING"] = "true"

# Define the LangSmith API endpoint.
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"

# Retrieve the LangSmith API key securely from Colab's secret store.
os.environ["LANGSMITH_API_KEY"] = userdata.get('LANGSMITH_API_KEY')

# Assign a project name to organize runs within LangSmith.
os.environ["LANGSMITH_PROJECT"] = "RAG PROJECT"

# Retrieve and set the OpenAI API key for embedding and LLM calls.
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

# Retrieve and set the Pinecone API key for database access.
os.environ["PINECONE_API_KEY"] = userdata.get('PINECONE_API_KEY')

# Specify the OpenAI embedding model (1536-dimensional output).
OPENAI_MODEL_EMB = "text-embedding-3-small"  # Common lightweight embedding model

# Define Pinecone configuration variables.
INDEX_NAME = "day3-rag-index"     # Vector index name in Pinecone
PINECONE_CLOUD = "aws"            # Cloud provider for Pinecone instance
PINECONE_REGION = "us-east-1"     # Region where the Pinecone index is hosted


🧠 Step 3: Import Core Libraries for RAG Pipeline

In this step, we import all the essential libraries needed for document ingestion, text splitting, embedding generation, and vector database management.

In [9]:
# 🧠 Step 3: Import Core Libraries for RAG Pipeline

# Standard utilities
from pathlib import Path    # To handle file and folder paths cleanly
import json                 # To read and parse JSON metadata files
from typing import List     # For type hints, ensuring readable and maintainable functions

# LangChain components for OpenAI integration
from langchain_openai import OpenAIEmbeddings, ChatOpenAI  # For embeddings and LLM access

# Document loaders to read PDFs and text files into LangChain documents
from langchain_community.document_loaders import PyPDFLoader, TextLoader

# Text splitter to chunk long documents into manageable parts for embeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Document schema defines how each document is represented internally
from langchain.schema import Document

# Pinecone setup and vector store integration
from pinecone import Pinecone, ServerlessSpec  # Pinecone SDK and configuration for serverless indexing
from langchain_pinecone import PineconeVectorStore  # LangChain wrapper for Pinecone vector database


#📂 Step 4: Load and Structure Core and Student Documents

In this step, we define utility functions to load and structure all the documents required for the RAG pipeline. This includes:

Loading core reference PDFs such as projects, criteria, and mentors.

Iterating through student folders to load individual reports, summaries, and metadata.

Tagging each document with relevant metadata (like source, student, and category) to support context-aware retrieval later.

In [10]:
# 📂 Step 4: Load and Structure Core and Student Documents

# Point this to your Google Drive or Colab directory where 'Day3_RAG' data is stored
DATA = Path("/content/drive/MyDrive/APU TRAINING/TRAINING DATA")

# Core PDFs that serve as global reference material
CORE_PDFS = [
    DATA / "Projects.pdf",
    DATA / "criteria.pdf",
    DATA / "Mentors.pdf",
]

def load_core_pdfs() -> List[Document]:
    """
    Loads all core PDF documents (projects, criteria, mentors).
    Each page is converted into a LangChain Document with metadata attached.
    """
    docs = []
    for pdf in CORE_PDFS:
        if pdf.exists():
            # Load all pages from the PDF
            pages = PyPDFLoader(str(pdf)).load()
            # Add source metadata for traceability
            for p in pages:
                p.metadata.update({"source": pdf.name, "category": "core"})
            docs.extend(pages)
    return docs


def load_student_dirs() -> List[Document]:
    """
    Iterates through each student directory and loads:
      - report.pdf: main project report
      - summary.txt: short text summary (if exists)
      - metadata.json: student metadata (if exists)
    Each document is wrapped in a LangChain Document object with descriptive metadata.
    """
    docs = []
    students_dir = DATA / "students"
    if not students_dir.exists():
        return docs

    for student_dir in students_dir.iterdir():
        if not student_dir.is_dir():
            continue

        # --- Load student report (PDF) ---
        report_pdf = student_dir / "report.pdf"
        if report_pdf.exists():
            pages = PyPDFLoader(str(report_pdf)).load()
            for p in pages:
                p.metadata.update({
                    "source": f"{student_dir.name}/report.pdf",
                    "student": student_dir.name,
                    "category": "student_report"
                })
            docs.extend(pages)

        # --- Load student summary (TXT) ---
        summary_txt = student_dir / "summary.txt"
        if summary_txt.exists():
            tdocs = TextLoader(str(summary_txt), encoding="utf-8").load()
            for d in tdocs:
                d.metadata.update({
                    "source": f"{student_dir.name}/summary.txt",
                    "student": student_dir.name,
                    "category": "student_summary"
                })
            docs.extend(tdocs)

        # --- Load student metadata (JSON) ---
        meta_json = student_dir / "metadata.json"
        if meta_json.exists():
            try:
                meta = json.loads(meta_json.read_text(encoding="utf-8"))
                meta_doc = Document(
                    page_content=json.dumps(meta, ensure_ascii=False, indent=2),
                    metadata={
                        "source": f"{student_dir.name}/metadata.json",
                        "student": student_dir.name,
                        "category": "student_metadata"
                    }
                )
                docs.append(meta_doc)
            except Exception as e:
                print(f"Could not parse {meta_json}: {e}")

    return docs


# Combine all loaded documents from core and student sources
raw_docs = load_core_pdfs() + load_student_dirs()

# Display how many total document objects were loaded
print(f"Loaded {len(raw_docs)} raw documents (pages + text).")


Loaded 134 raw documents (pages + text).


#✂️ Step 5: Split Documents into Manageable Chunks

In this step, we use a recursive character text splitter to break down long documents into smaller, overlapping chunks.
This ensures that each chunk is within token limits and still preserves contextual continuity for embeddings and retrieval.

chunk_size=1000: Maximum number of characters in a single chunk.

chunk_overlap=150: Overlap between consecutive chunks to maintain context across boundaries.

separators: Defines the preferred order of splitting (paragraphs → lines → words → characters).

Finally, we print the number of generated chunks and preview one sample with its metadata for verification.

In [11]:
# ✂️ Step 5: Split Documents into Manageable Chunks

# Initialize the RecursiveCharacterTextSplitter with chunking parameters.
# It prioritizes larger splits first (paragraphs, then sentences, etc.)
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,                 # Maximum size per chunk in characters
    chunk_overlap=150,               # Overlap to preserve context
    separators=["\n\n", "\n", " ", ""],  # Logical split hierarchy (paragraph > line > word > char)
)

# Apply the splitter to the combined raw documents
chunks = splitter.split_documents(raw_docs)

# Display how many text chunks were created
print(f"Created {len(chunks)} chunks.")

# Preview the first chunk to verify splitting and metadata tagging
print(chunks[0].page_content[:300], "...\n", chunks[0].metadata)


Created 287 chunks.
UNIVERSITY
 
PROJECT
 
GUIDELINES
 
2024
 
 
CAPSTONE
 
PROJECT
 
REQUIREMENTS
 
 
1.
 
PROJECT
 
DURATION
 
AND
 
MILESTONES
 
   
-
 
Project
 
Duration:
 
12
 
weeks
 
minimum
 
   
-
 
Proposal
 
Submission:
 
Week
 
1
 
   
-
 
Mid-term
 
Review:
 
Week
 
6
 
   
-
 
Final
 
Submission:
 
Week
 ...
 {'source': 'Projects.pdf', 'page': 0, 'category': 'core'}


#🧮 Step 6: Generate Embeddings and Store Them in Pinecone

In this step, we initialize the embedding model and Pinecone vector database, then index all text chunks for retrieval.
This forms the foundation of the RAG system — enabling semantic search and contextual grounding.

Embeddings: We use OpenAI’s text-embedding-3-small (1536-dim) to convert text chunks into dense numerical vectors.

Pinecone: A high-performance vector database for storing and searching embeddings using cosine similarity.

LangChain Vector Store: Wraps Pinecone to simplify storing and retrieving document embeddings.

In [12]:
# 🧮 Step 6: Generate Embeddings and Store Them in Pinecone

# Initialize OpenAI embedding model (1536-dimension for text-embedding-3-small)
embeddings = OpenAIEmbeddings(model=OPENAI_MODEL_EMB)

# Initialize the Pinecone client using the API key from environment variables
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

# Create a new index if it doesn't exist already
# The index is configured for cosine similarity matching
if INDEX_NAME not in [ix["name"] for ix in pc.list_indexes()]:
    pc.create_index(
        name=INDEX_NAME,                 # Index name defined earlier
        dimension=1536,                  # Must match embedding model’s output dimension
        metric="cosine",                 # Cosine distance for semantic similarity
        spec=ServerlessSpec(
            cloud=PINECONE_CLOUD,        # Cloud provider (aws/gcp)
            region=PINECONE_REGION       # Specific region for hosting
        ),
    )

# Build a LangChain vector store wrapper on top of the Pinecone index
# This allows seamless integration between LangChain documents and Pinecone embeddings
vectorstore = PineconeVectorStore.from_documents(
    documents=chunks,                    # Chunked documents to be embedded
    embedding=embeddings,                # Embedding model used
    index_name=INDEX_NAME,               # Pinecone index name
)


#💬 Step 7: Build and Run the RAG Question-Answering Chain

In this step, we connect the retriever, language model, and prompt into a working Retrieval-Augmented Generation (RAG) pipeline.
The retriever fetches the most relevant chunks from Pinecone, and the LLM uses them as context to answer the user’s question.

Key Components:

LLM: A lightweight OpenAI chat model (gpt-4o-mini) is used for low-cost, fast inference.

Retriever: Fetches the top-k (here k=4) semantically closest chunks from the vector store.

PromptTemplate: Guides the LLM to answer only using the retrieved context, avoiding hallucination.

Chain: Combines retrieval, prompt formatting, and LLM inference into a single callable pipeline.

At the end, a sample question is asked, and both the final answer and the retrieved chunks are displayed to show how the model grounded its response.

In [14]:
# 💬 Step 7: Build and Run the RAG Question-Answering Chain

# Initialize the LLM — small and cost-effective model for demo purposes
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)  # deterministic responses

# Create a retriever from the Pinecone vector store
# It fetches the top 4 most relevant chunks per query
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# Import utilities for prompt-based LLM chaining
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema import StrOutputParser

# Define a strict prompt template that instructs the model
# to answer solely based on the retrieved context
prompt = PromptTemplate.from_template(
    "You are a helpful assistant. Use ONLY the provided context to answer.\n\n"
    "Question: {question}\n\n"
    "Context:\n{context}\n\n"
    "Answer:"
)

# Helper to format retrieved docs with their metadata for readability
def format_docs(docs):
    return "\n\n".join(
        [f"[{i+1}] {d.page_content}\n(meta: {d.metadata})" for i, d in enumerate(docs)]
    )

# Create the full retrieval-augmented generation chain
chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Example question that may match multiple documents
question = "Provide me name of 2 students who can collaborate on a Robot and Health domain project"

# Run the RAG chain
answer = chain.invoke(question)
print("ANSWER:\n", answer)

# Retrieve and display the raw context chunks for transparency / debugging
docs = retriever.get_relevant_documents(question)
print("\n--- Retrieved Chunks (for debugging) ---")
for i, d in enumerate(docs, 1):
    print(f"[{i}] from {d.metadata.get('source')}, p={d.metadata.get('page', 'NA')}")
    print(d.page_content[:300].strip(), "...\n")


ANSWER:
 Kenji Tanaka and Aisha Khan can collaborate on a Robot and Health domain project.

--- Retrieved Chunks (for debugging) ---
[1] from Mentors.pdf, p=0.0
FACULTY
 
MENTORS
 
AND
 
AREAS
 
OF
 
EXPERTISE
 
2024
 
 
COMPUTER
 
SCIENCE
 
DEPARTMENT:
 
Dr.
 
Sarah
 
Chen
 
-
 
Artificial
 
Intelligence
 
in
 
Healthcare
 
-
 
Computer
 
Vision
 
-
 
Medical
 
Image
 
Processing
 
Office:
 
CS
 
Building
 
301
 
Email:
 
schen@university.edu
 
 
Dr.
 
Mic ...

[2] from Kenji/metadata.json, p=NA
{
  "student_id": "STU2024004",
  "name": "Kenji Tanaka",
  "email": "kenji.tanaka@university.edu",
  "department": "Robotics Engineering",
  "project_title": "Autonomous Delivery Robot with Obstacle Avoidance",
  "submission_date": "2024-01-22",
  "academic_year": "Second Year",
  "supervisor": "Dr ...

[3] from Projects.pdf, p=0.0
-
 
Code
 
repository
 
with
 
documentation
 
   
-
 
Test
 
cases
 
and
 
validation
 
results
 
   
-
 
User
 
manual
 
(if
 
applicable)
 
 
4.
 
EVALUATION
 
